<a href="https://colab.research.google.com/github/Gaddy01/Time-Series-Forcasting/blob/main/01_Data_Handling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1. Data Handling and Memory Management

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import json
import gc
import os
import psutil
import shutil
from datetime import datetime

import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# PROJECT CONFIGURATION

BASE_DIR = Path("/content/formative1")

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = BASE_DIR / "results"

FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
LOGS_DIR = RESULTS_DIR / "logs"

for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    LOGS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directories ready.")

Project directories ready.


In [ ]:
# DATAVERSE CONFIGURATION

DATAVERSE_BASE_URL = "https://dataverse.harvard.edu"
DATASET_PID = "doi:10.7910/DVN/EGZHFV"
DATASET_API_URL = (f"{DATAVERSE_BASE_URL}/api/datasets/:persistentId/")

# Your own information
GUESTBOOK_RESPONSE = {
    "guestbookResponse": {
        "name": "YOUR NAME",
        "email": "YOUR EMAIL",
        "institution": "African Leadership University",
        "position": "Student"
    }
}

# Chunk size for pandas processing
CHUNK_SIZE = 100_000

# Test mode:
# True  -> process only the two files we currently have
# False -> process all 62 files
TEST_MODE = True

print("Configuration loaded.")

Configuration loaded.


In [ ]:
# GET DATASET FILE METADATA

url = "https://dataverse.harvard.edu/api/datasets/:persistentId/"
params = {"persistentId": "doi:10.7910/DVN/EGZHFV"}
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/131.0 Safari/537.36"
}

response = requests.get(
    url,
    params=params,
    headers=headers
)

print("Status:", response.status_code)
print(response.text[:1000])

Status: 200
{"status":"OK","data":{"id":2674254,"identifier":"DVN/EGZHFV","persistentUrl":"https://doi.org/10.7910/DVN/EGZHFV","protocol":"doi","authority":"10.7910","separator":"/","publisher":"Harvard Dataverse","publicationDate":"2015-05-14","storageIdentifier":"s3://10.7910/DVN/EGZHFV","guestbookId":96,"effectiveDatasetFileCountLimit":1000,"datasetFileUploadsAvailable":938,"datasetType":"dataset","locks":[],"latestVersion":{"id":181170,"datasetId":2674254,"datasetPersistentId":"doi:10.7910/DVN/EGZHFV","datasetType":"dataset","storageIdentifier":"s3://10.7910/DVN/EGZHFV","versionNumber":1,"internalVersionNumber":4,"versionMinorNumber":3,"versionState":"RELEASED","latestVersionPublishingState":"RELEASED","deaccessionLink":"","lastUpdateTime":"2020-02-09T11:14:03Z","releaseTime":"2020-02-09T11:14:03Z","createTime":"2020-02-09T11:13:14Z","publicationDate":"2015-05-14","citationDate":"2015-05-14","guestbookId":96,"effectiveDatasetFileCountLimit":1000,"datasetFileUploadsAvailable":938,"t

In [ ]:
dataset_data = response.json()["data"]
files = dataset_data["latestVersion"]["files"]
print(f"Number of files: {len(files)}")

Number of files: 62


In [ ]:
# BUILD FILE METADATA TABLE

file_records = []

for file_entry in files:
    data_file = file_entry["dataFile"]

    file_records.append({
        "filename": data_file["filename"],
        "file_id": data_file["id"],
        "filesize_bytes": data_file["filesize"],
        "filesize_MB": data_file["filesize"] / (1024 ** 2),
        "filesize_GB": data_file["filesize"] / (1024 ** 3),
    })

files_df = pd.DataFrame(file_records)

files_df = (
    files_df
    .sort_values("filename")
    .reset_index(drop=True)
)

files_df["download_url"] = (
    "https://dataverse.harvard.edu/api/access/datafile/"
    + files_df["file_id"].astype(str)
)

print("Number of files:", len(files_df))
display(files_df.head())

Number of files: 62


,filename,file_id,filesize_bytes,filesize_MB,filesize_GB,download_url
0,sms-call-internet-mi-2013-11-01.txt,2674255,322874887,307.917487,0.300701,https://dataverse.harvard.edu/api/access/dataf...
1,sms-call-internet-mi-2013-11-02.txt,2674265,315164060,300.563869,0.293519,https://dataverse.harvard.edu/api/access/dataf...
2,sms-call-internet-mi-2013-11-03.txt,2674273,313058344,298.555702,0.291558,https://dataverse.harvard.edu/api/access/dataf...
3,sms-call-internet-mi-2013-11-04.txt,2674282,361097689,344.369592,0.336298,https://dataverse.harvard.edu/api/access/dataf...
4,sms-call-internet-mi-2013-11-05.txt,2674279,372894401,355.619813,0.347285,https://dataverse.harvard.edu/api/access/dataf...


Let's check the overall size of the dataset and the top 5 largest files

In [ ]:
print("Total size:")
print(f"{files_df['filesize_GB'].sum():.2f} GB")

print("\nLargest files:")
display(
    files_df.nlargest(5, "filesize_MB")[
        ["filename", "file_id", "filesize_MB", "filesize_GB"]
    ]
)

Total size:
19.38 GB

Largest files:


,filename,file_id,filesize_MB,filesize_GB
7,sms-call-internet-mi-2013-11-08.txt,2674261,358.698951,0.350292
5,sms-call-internet-mi-2013-11-06.txt,2674283,357.271122,0.348898
6,sms-call-internet-mi-2013-11-07.txt,2674271,355.755124,0.347417
4,sms-call-internet-mi-2013-11-05.txt,2674279,355.619813,0.347285
14,sms-call-internet-mi-2013-11-15.txt,2674281,347.914138,0.339760


Now, let's save the metadata table so you don't have to query Dataverse again

In [ ]:
metadata_path = PROCESSED_DIR / "dataverse_file_metadata.csv"

files_df[
    [
        "filename",
        "file_id",
        "filesize_bytes",
        "filesize_MB",
        "filesize_GB",
        "download_url"
    ]
].to_csv(metadata_path, index=False)

print(f"Saved metadata to: {metadata_path}")

Saved metadata to: /content/formative1/data/processed/dataverse_file_metadata.csv


Let's now define the dataset columns. From my inspection of the actual file, I know the raw file has eight tab-separated columns.

In [ ]:
# Original raw dataset columns
COLUMNS = [
    "square_id",
    "timestamp",
    "country_code",
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet"
]

USECOLS = [
    "square_id",
    "timestamp",
    "country_code",
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet"
]

# Memory-efficient data types
DTYPES = {
    "square_id": "int16",
    "timestamp": "int64",
    "country_code": "int16",
    "sms_in": "float32",
    "sms_out": "float32",
    "call_in": "float32",
    "call_out": "float32",
    "internet": "float32"
}

Next, let's create our Memory measurement function

In [ ]:
# MEMORY MONITORING

process = psutil.Process(os.getpid())

def get_memory_mb():
    """Return current Python process RAM usage in MB."""
    return process.memory_info().rss / (1024 ** 2)


def get_system_memory():
    """Return system-level memory information."""
    memory = psutil.virtual_memory()

    return {
        "total_GB": memory.total / (1024 ** 3),
        "available_GB": memory.available / (1024 ** 3),
        "used_GB": memory.used / (1024 ** 3),
        "percent": memory.percent
    }


print(f"Current process RAM: {get_memory_mb():.2f} MB")
print("System memory:", get_system_memory())

Current process RAM: 241.03 MB
System memory: {'total_GB': 12.671417236328125, 'available_GB': 10.994491577148438, 'used_GB': 1.3952865600585938, 'percent': 13.2}


Next, let's create the Function to submit the Guestbook response

In [ ]:
# REQUEST SIGNED DOWNLOAD URL

GUESTBOOK_RESPONSE = {
    "guestbookResponse": {
        "name": "Gaddiel Irakoze",
        "email": "g.irakoze2@alustudent.com",
        "institution": "African Leadership University",
        "position": "Student"
    }
}

def get_signed_url(file_id):
    """
    Submit the Dataverse Guestbook response and
    obtain a temporary signed download URL.
    """

    url = f"{DATAVERSE_BASE_URL}/api/access/datafile/{file_id}"

    response = requests.post(
        url,
        json=GUESTBOOK_RESPONSE,
        timeout=60
    )

    response.raise_for_status()
    result = response.json()

    if result.get("status") != "OK":
        raise RuntimeError(
            f"Dataverse error for file {file_id}: {result}"
        )

    signed_url = result["data"]["signedUrl"]

    return signed_url

The following is the function to download one file

In [ ]:
# STREAM DOWNLOAD

def download_file(file_id, filename, output_path):
    """Request a fresh signed URL and stream the file to disk."""

    signed_url = get_signed_url(file_id)
    start_time = time.perf_counter()

    response = requests.get(
        signed_url,
        stream=True,
        timeout=120
    )

    response.raise_for_status()
    bytes_downloaded = 0

    with open(output_path, "wb") as f:

        for chunk in response.iter_content(
            chunk_size=1024 * 1024
        ):
            if chunk:
                f.write(chunk)
                bytes_downloaded += len(chunk)

    download_time = time.perf_counter() - start_time

    return {
        "bytes_downloaded": bytes_downloaded,
        "download_time_seconds": download_time
    }

Let's now implement a function that processed the downloaded raw file.

In [ ]:
# 8. Memory-Efficient Chunk Processing

def process_raw_file(raw_path):
    """
    Process one raw daily file in chunks.

    For each chunk:
    1. Read only the required columns.
    2. Aggregate activity across country codes.
    3. Keep one observation per square and timestamp.

    Returns:
        processed_df
        processing_time
        peak_memory_mb
    """

    aggregated_chunks = []

    peak_memory_mb = get_memory_mb()
    start_time = time.perf_counter()

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            raw_path,
            sep="\t",
            header=None,
            names=COLUMNS,
            usecols=USECOLS,
            dtype=DTYPES,
            chunksize=CHUNK_SIZE
        ),
        start=1
    ):

        # Aggregate telecommunications activity across country codes
        # for each geographical square and time interval.
        chunk_agg = (
            chunk
            .groupby(
                ["square_id", "timestamp"],
                as_index=False
            )[
                [
                    "sms_in",
                    "sms_out",
                    "call_in",
                    "call_out",
                    "internet"
                ]
            ]
            .sum(min_count=1)
        )

        aggregated_chunks.append(chunk_agg)

        # Track peak memory usage
        current_memory_mb = get_memory_mb()
        peak_memory_mb = max(
            peak_memory_mb,
            current_memory_mb
        )

        # Release the current chunk before reading the next one
        del chunk
        gc.collect()

    # Combine the aggregated chunks
    result = pd.concat(
        aggregated_chunks,
        ignore_index=True
    )

    # A square/timestamp combination may appear in more than one
    # chunk, so aggregate one final time.
    result = (
        result
        .groupby(
            ["square_id", "timestamp"],
            as_index=False
        )[
            [
                "sms_in",
                "sms_out",
                "call_in",
                "call_out",
                "internet"
            ]
        ]
        .sum(min_count=1)
    )

    # Sort for easier downstream time-series analysis
    result = result.sort_values(
        ["square_id", "timestamp"]
    ).reset_index(drop=True)

    processing_time = time.perf_counter() - start_time

    return (
        result,
        processing_time,
        peak_memory_mb
    )

Let's now save and verify the processed data

In [ ]:
# ============================================================
# 9. Save Processed Data
# ============================================================

def save_processed_data(df, filename):
    """
    Save the aggregated daily dataset.

    The processed dataset contains: square_id, timestamp, sms_in
        sms_out, call_in, call_out, internet
    """

    output_filename = filename.replace(
        ".txt",
        "_aggregated.csv"
    )

    output_path = PROCESSED_DIR / output_filename

    # Save processed data
    df.to_csv(
        output_path,
        index=False
    )

    # Verify that the output was created successfully
    if not output_path.exists():
        raise RuntimeError(
            f"Processed file was not created: {output_path}"
        )

    if output_path.stat().st_size == 0:
        raise RuntimeError(
            f"Processed file is empty: {output_path}"
        )

    return output_path

Next, let's make our pipeline resumable.

In [ ]:
# PROCESSING LOG

LOG_PATH = LOGS_DIR / "processing_log.csv"

LOG_COLUMNS = [
    "filename",
    "file_id",
    "status",
    "raw_size_bytes",
    "download_time_seconds",
    "processing_time_seconds",
    "peak_memory_mb",
    "processed_rows",
    "processed_size_bytes",
    "error"
]

def load_processing_log():
    if LOG_PATH.exists():
        return pd.read_csv(LOG_PATH)
    return pd.DataFrame(columns=LOG_COLUMNS)

def save_processing_log(log_df):
    log_df.to_csv(
        LOG_PATH,
        index=False
    )

Finally, let's put everything together.

In [ ]:
# PROCESS ONE DATASET FILE

def process_dataset_file(file_row):

    filename = file_row["filename"]
    file_id = int(file_row["file_id"])
    expected_size = int(file_row["filesize_bytes"])

    raw_path = RAW_DIR / filename

    print("\n" + "=" * 70)
    print(f"Processing: {filename}")
    print(f"File ID: {file_id}")
    print("=" * 70)

    # Download
    print("Requesting fresh signed URL and downloading...")

    download_info = download_file(
        file_id,
        filename,
        raw_path
    )

    actual_size = raw_path.stat().st_size

    print(
        f"Downloaded: "
        f"{actual_size / (1024 ** 2):.2f} MB"
    )

    # Verify download size
    if actual_size != expected_size:

        raise RuntimeError(
            f"Size mismatch for {filename}: "
            f"expected {expected_size}, "
            f"got {actual_size}"
        )

    print("Download verification: PASSED")

    # Process
    print("Processing in chunks...")

    processed_df, processing_time, peak_memory = process_raw_file(raw_path)

    print(f"Processing time: {processing_time:.2f} seconds")
    print(f"Peak process RAM: {peak_memory:.2f} MB")
    print(f"Aggregated rows: {len(processed_df):,}")

    processed_rows = len(processed_df)

    # Save processed data
    output_path = save_processed_data(processed_df, filename)

    print(f"Saved: {output_path}")
    processed_size = output_path.stat().st_size

    # Delete raw file ONLY after successful processing
    del processed_df
    gc.collect()

    raw_path.unlink()

    print("Raw file deleted.")

    return {
        "filename": filename,
        "file_id": file_id,
        "status": "completed",
        "raw_size_bytes": actual_size,
        "download_time_seconds": download_info[
            "download_time_seconds"
        ],
        "processing_time_seconds": processing_time,
        "peak_memory_mb": peak_memory,
        "processed_rows": processed_rows,
        "processed_size_bytes": processed_size,
        "error": ""
    }

## Running ....

Let's first test it on the two files

In [ ]:
# SELECT TEST FILES

TEST_FILE_IDS = [2674255, 2674265]

test_files_df = files_df[
    files_df["file_id"].isin(TEST_FILE_IDS)
].copy()

display(
    test_files_df[
        ["filename", "file_id", "filesize_MB"]
    ]
)

,filename,file_id,filesize_MB
0,sms-call-internet-mi-2013-11-01.txt,2674255,307.917487
1,sms-call-internet-mi-2013-11-02.txt,2674265,300.563869


In [ ]:
# RUN TEST PIPELINE

processing_log = load_processing_log()

for _, file_row in test_files_df.iterrows():

    filename = file_row["filename"]

    # Skip if already completed
    already_completed = (
        (processing_log["filename"] == filename)
        &
        (processing_log["status"] == "completed")
    ).any()

    if already_completed:
        print(
            f"\nSkipping {filename} "
            f"(already completed)."
        )
        continue

    try:
        result = process_dataset_file(file_row)
        processing_log = pd.concat(
            [
                processing_log,
                pd.DataFrame([result])
            ],
            ignore_index=True
        )

        save_processing_log(processing_log)

    except Exception as e:
        error_result = {
            "filename": filename,
            "file_id": int(file_row["file_id"]),
            "status": "failed",
            "raw_size_bytes": 0,
            "download_time_seconds": 0,
            "processing_time_seconds": 0,
            "peak_memory_mb": 0,
            "processed_rows": 0,
            "processed_size_bytes": 0,
            "error": str(e)
        }

        processing_log = pd.concat(
            [
                processing_log,
                pd.DataFrame([error_result])
            ],
            ignore_index=True
        )

        save_processing_log(processing_log)

        print(f"ERROR: {e}")


Processing: sms-call-internet-mi-2013-11-01.txt
File ID: 2674255
Requesting fresh signed URL and downloading...
Downloaded: 307.92 MB
Download verification: PASSED
Processing in chunks...
Processing time: 12.69 seconds
Peak process RAM: 287.75 MB
Aggregated rows: 1,439,982
Saved: /content/formative1/data/processed/sms-call-internet-mi-2013-11-01_aggregated.csv
Raw file deleted.

Processing: sms-call-internet-mi-2013-11-02.txt
File ID: 2674265
Requesting fresh signed URL and downloading...


/tmp/ipykernel_7539/3283549550.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  processing_log = pd.concat(


Downloaded: 300.56 MB
Download verification: PASSED
Processing in chunks...
Processing time: 11.48 seconds
Peak process RAM: 286.37 MB
Aggregated rows: 1,439,986
Saved: /content/formative1/data/processed/sms-call-internet-mi-2013-11-02_aggregated.csv
Raw file deleted.


## Running the processing of the remaining 60

In [ ]:
# Process Remaining Dataset Files

processing_log = load_processing_log()

# Files that have already been successfully processed
completed_files = set(
    processing_log.loc[
        processing_log["status"] == "completed",
        "filename"
    ]
)

# Determine which files still need to be processed
remaining_files_df = files_df[
    ~files_df["filename"].isin(completed_files)
].copy()

print("=" * 70)
print("FULL DATASET PROCESSING")
print("=" * 70)

print(f"Total files in dataset: {len(files_df)}")
print(f"Already completed:      {len(completed_files)}")
print(f"Remaining to process:   {len(remaining_files_df)}")
print("=" * 70)

# Process remaining files one at a time
for _, file_row in remaining_files_df.iterrows():
    filename = file_row["filename"]
    #print(f"\nFiles remaining: {len(remaining_files_df)}")

    try:
        result = process_dataset_file(file_row)

        # Add successful result to the log
        processing_log = pd.concat(
            [
                processing_log,
                pd.DataFrame([result])
            ],
            ignore_index=True
        )

        save_processing_log(processing_log)
        print("STATUS: COMPLETED")

    except Exception as e:
        # Record failed file
        error_result = {
            "filename": filename,
            "file_id": int(file_row["file_id"]),
            "status": "failed",
            "raw_size_bytes": int(file_row["filesize_bytes"]),
            "download_time_seconds": 0,
            "processing_time_seconds": 0,
            "peak_memory_mb": 0,
            "processed_rows": 0,
            "processed_size_bytes": 0,
            "error": str(e)
        }

        processing_log = pd.concat(
            [
                processing_log,
                pd.DataFrame([error_result])
            ],
            ignore_index=True
        )

        save_processing_log(processing_log)
        print(f"STATUS: FAILED")
        print(f"ERROR: {e}")
        continue # Continue with the next file instead of stopping

print("\n" + "=" * 70)
print("PROCESSING RUN FINISHED")
print("=" * 70)

processing_log = load_processing_log()

print("Completed:", (processing_log["status"] == "completed").sum())
print("Failed:", (processing_log["status"] == "failed").sum())
print("Total logged:", len(processing_log))

FULL DATASET PROCESSING
Total files in dataset: 62
Already completed:      5
Remaining to process:   57

Processing: sms-call-internet-mi-2013-11-06.txt
File ID: 2674283
Requesting fresh signed URL and downloading...
Downloaded: 357.27 MB
Download verification: PASSED
Processing in chunks...
Processing time: 14.66 seconds
Peak process RAM: 288.20 MB
Aggregated rows: 1,439,980
Saved: /content/formative1/data/processed/sms-call-internet-mi-2013-11-06_aggregated.csv
Raw file deleted.
STATUS: COMPLETED

Processing: sms-call-internet-mi-2013-11-07.txt
File ID: 2674271
Requesting fresh signed URL and downloading...
Downloaded: 355.76 MB
Download verification: PASSED
Processing in chunks...
Processing time: 15.36 seconds
Peak process RAM: 292.65 MB
Aggregated rows: 1,439,981
Saved: /content/formative1/data/processed/sms-call-internet-mi-2013-11-07_aggregated.csv
Raw file deleted.
STATUS: COMPLETED

Processing: sms-call-internet-mi-2013-11-08.txt
File ID: 2674261
Requesting fresh signed URL an

Since the final dataset will contain roughly 89 million rows, we should **Parquet** the main merged file. It will be substantially smaller than CSV and much faster to load for EDA/modeling.

In [ ]:
# Merge All Processed Daily Files into One Master Parquet

PROCESSED_DIR = Path("/content/formative1/data/processed")
MASTER_DIR = PROCESSED_DIR.parent / "master"
MASTER_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PARQUET = MASTER_DIR / "telecom_milan_master.parquet"

processed_files = sorted(
    PROCESSED_DIR.glob("*_aggregated.csv")
)

print("=" * 70)
print("MERGING PROCESSED DATASETS")
print("=" * 70)
print(f"Files found: {len(processed_files)}")
print(f"Output: {MASTER_PARQUET}")
print("=" * 70)

if len(processed_files) != 62:
    raise RuntimeError(
        f"Expected 62 processed files, but found "
        f"{len(processed_files)}."
    )

# Remove an existing master file so that the merge starts cleanly
if MASTER_PARQUET.exists():
    MASTER_PARQUET.unlink()
    print("Existing master file removed.")

# Compact data types
DTYPES_MASTER = {
    "square_id": "int16",
    "timestamp": "int64",
    "sms_in": "float32",
    "sms_out": "float32",
    "call_in": "float32",
    "call_out": "float32",
    "internet": "float32"
}

COLUMNS_MASTER = [
    "square_id",
    "timestamp",
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet"
]

writer = None
total_rows = 0
start_time = time.perf_counter()

try:

    for position, file_path in enumerate(
        processed_files,
        start=1
    ):

        print(
            f"\nProcessing file {position}/{len(processed_files)}: "
            f"{file_path.name}"
        )

        # Read one daily file at a time
        df = pd.read_csv(
            file_path,
            usecols=COLUMNS_MASTER,
            dtype=DTYPES_MASTER
        )

        # Convert pandas DataFrame to PyArrow table
        table = pa.Table.from_pandas(df,preserve_index=False)

        # Create the Parquet writer using the first file's schema
        if writer is None:
            writer = pq.ParquetWriter(
                MASTER_PARQUET,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)

        rows = len(df)
        total_rows += rows

        print(
            f"Rows added: {rows:,} | "
            f"Total rows: {total_rows:,}"
        )

        del df
        del table
        gc.collect()

finally:

    if writer is not None:
        writer.close()

elapsed_time = time.perf_counter() - start_time

print("\n" + "=" * 70)
print("MERGE FINISHED")
print("=" * 70)
print(f"Files merged: {len(processed_files)}")
print(f"Total rows:   {total_rows:,}")
print(f"Time:         {elapsed_time:.2f} seconds")
print(
    f"Output size:  "
    f"{MASTER_PARQUET.stat().st_size / (1024**3):.2f} GB"
)
print(f"Output file:  {MASTER_PARQUET}")
print("=" * 70)

MERGING PROCESSED DATASETS
Files found: 62
Output: /content/formative1/data/master/telecom_milan_master.parquet

Processing file 1/62: sms-call-internet-mi-2013-11-01_aggregated.csv
Rows added: 1,439,982 | Total rows: 1,439,982

Processing file 2/62: sms-call-internet-mi-2013-11-02_aggregated.csv
Rows added: 1,439,986 | Total rows: 2,879,968

Processing file 3/62: sms-call-internet-mi-2013-11-03_aggregated.csv
Rows added: 1,439,963 | Total rows: 4,319,931

Processing file 4/62: sms-call-internet-mi-2013-11-04_aggregated.csv
Rows added: 1,439,976 | Total rows: 5,759,907

Processing file 5/62: sms-call-internet-mi-2013-11-05_aggregated.csv
Rows added: 1,439,977 | Total rows: 7,199,884

Processing file 6/62: sms-call-internet-mi-2013-11-06_aggregated.csv
Rows added: 1,439,980 | Total rows: 8,639,864

Processing file 7/62: sms-call-internet-mi-2013-11-07_aggregated.csv
Rows added: 1,439,981 | Total rows: 10,079,845

Processing file 8/62: sms-call-internet-mi-2013-11-08_aggregated.csv
Rows 

Let's now perform validation on the master dataset.

However, we will not load the entire 1.74 GB Parquet file into a pandas DataFrame just for validation. The resulting in-memory DataFrame could be several GB, and there's no reason to take that risk.

Instead, let's perform a memory-efficient validation using PyArrow. It can inspect the Parquet metadata and process the columns in batches.

In [ ]:
# Memory-Efficient Master Dataset Validation

print("=" * 70)
print("MASTER DATASET VALIDATION")
print("=" * 70)

parquet_file = pq.ParquetFile(MASTER_PARQUET)

# Basic metadata

print("\nParquet metadata:")
print(f"Row groups: {parquet_file.num_row_groups:,}")
print(f"Total rows: {parquet_file.metadata.num_rows:,}")
print(f"Columns:    {parquet_file.metadata.num_columns}")

print("\nSchema:")
print(parquet_file.schema_arrow)

# Validate data in batches

columns = [
    "square_id",
    "timestamp",
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet"
]

total_rows = 0
total_duplicates = 0
min_timestamp = None
max_timestamp = None

unique_squares = set()

missing_counts = {
    column: 0
    for column in columns
}

start_time = time.perf_counter()

for batch_number, batch in enumerate(
    parquet_file.iter_batches(
        batch_size=500_000,
        columns=columns
    ),
    start=1
):

    df = batch.to_pandas()

    total_rows += len(df)

    # Unique geographical areas
    unique_squares.update(
        df["square_id"].unique()
    )

    # Missing values
    for column in columns:
        missing_counts[column] += int(
            df[column].isna().sum()
        )

    # Timestamp range
    batch_min = df["timestamp"].min()
    batch_max = df["timestamp"].max()

    if min_timestamp is None or batch_min < min_timestamp:
        min_timestamp = batch_min

    if max_timestamp is None or batch_max > max_timestamp:
        max_timestamp = batch_max

    del df

    if batch_number % 20 == 0:
        print(
            f"Validated batches: {batch_number} | "
            f"Rows checked: {total_rows:,}"
        )

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

print(f"Total rows:       {total_rows:,}")
print(f"Unique squares:   {len(unique_squares):,}")

print(
    f"Timestamp start:  "
    f"{pd.to_datetime(min_timestamp, unit='ms')}"
)

print(
    f"Timestamp end:    "
    f"{pd.to_datetime(max_timestamp, unit='ms')}"
)

print("\nMissing values:")
for column, count in missing_counts.items():
    print(f"{column:12s}: {count:,}")

print(f"\nValidation time: {elapsed:.2f} seconds")

print("=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

MASTER DATASET VALIDATION

Parquet metadata:
Row groups: 124
Total rows: 89,245,318
Columns:    7

Schema:
square_id: int16
timestamp: int64
sms_in: float
sms_out: float
call_in: float
call_out: float
internet: float
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 900
Validated batches: 20 | Rows checked: 10,000,000
Validated batches: 40 | Rows checked: 20,000,000
Validated batches: 60 | Rows checked: 30,000,000
Validated batches: 80 | Rows checked: 40,000,000
Validated batches: 100 | Rows checked: 50,000,000
Validated batches: 120 | Rows checked: 60,000,000
Validated batches: 140 | Rows checked: 70,000,000
Validated batches: 160 | Rows checked: 80,000,000

VALIDATION RESULTS
Total rows:       89,245,318
Unique squares:   10,000
Timestamp start:  2013-10-31 23:00:00
Timestamp end:    2014-01-01 22:50:00

Missing values:
square_id   : 0
timestamp   : 0
sms_in      : 8,862,250
sms_out     : 9,132,482
call_in     : 15,149,421
call_out    

Because I want that next time, I can be able to reconnect Google Drive and continue from the master dataset without downloading/processsing the 20 GB raw dataset again, I will store all the necessary documents on my drive.

In [ ]:
# First, mount Drive:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create a project folder

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/ML Techniques I/Formative 1"
)

DRIVE_DATA = DRIVE_PROJECT / "data"
DRIVE_RESULTS = DRIVE_PROJECT / "results"

DRIVE_DATA.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

print(DRIVE_PROJECT)

/content/drive/MyDrive/ML Techniques I/Formative 1


In [ ]:
# Copy the master Parquet

master_destination = (
    DRIVE_DATA / "telecom_milan_master.parquet"
)

shutil.copy2(
    MASTER_PARQUET,
    master_destination
)

print("Master dataset copied to:")
print(master_destination)

Master dataset copied to:
/content/drive/MyDrive/ML Techniques I/Formative 1/data/telecom_milan_master.parquet


In [ ]:
# Copy the processing log
processing_log_source = (
    Path("/content/formative1/results/logs")
    / "processing_log.csv"
)

shutil.copy2(
    processing_log_source,
    DRIVE_RESULTS / "processing_log.csv"
)

print("Processing log copied.")

# Copy the metadata
metadata_source = (
    Path("/content/formative1/data/processed")
    / "dataverse_file_metadata.csv"
)

shutil.copy2(
    metadata_source,
    DRIVE_DATA / "dataverse_file_metadata.csv"
)

print("Dataverse metadata copied.")

Processing log copied.
Dataverse metadata copied.


In [ ]:
print(f"{master_destination.stat().st_size / (1024**3):.2f} GB")

1.74 GB
